# Model Training (reference data are videos)

In [30]:
filename = 'ADTC_8_adz_raw_20260304'

In [31]:
from video_download import video_downloader

datum = video_downloader(filename)

Connection with geo-amberg.ch
Successful access to /cam/2026/03/04
ADTC Rueschlikon_00_20260304165950.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304011636.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304203110.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304215925.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304215201.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304163643.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304011809.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304192522.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304021020.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304225913.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304205433.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304015054.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304020010.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304020027.mp4 successfully downloaded
ADTC Rueschlikon_00_20260304230021.mp4 successfully

## DataFrame for each video (path and timestamp)

In [32]:
import os
import pandas as pd
from pathlib import Path

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

src = root / f'VIDEO/{datum}'

rows = []

for f in os.listdir(src):
    if f.endswith('.mp4'):
        file = os.path.join(src, f)
        t = f[-18:-1]
        year = t[0:4]
        month = t[4:6]
        day = t[6:8]
        hour = t[8:10]
        minute = t[10:12]
        seconde = t[12:14]

        timestamp_video = f'{year}-{month}-{day} {hour}:{minute}:{seconde}'
        
        rows.append({
            "video": file,
            "timestamp_video": pd.to_datetime(timestamp_video)
        })

df_video = pd.DataFrame(rows)
df_video.head()

,video,timestamp_video
0,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 00:35:56
1,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 00:36:13
2,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 00:38:22
3,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 00:38:41
4,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 00:39:45


In [33]:
df_train_data = pd.read_csv(f'DataFrame/df_merged_{filename}.csv')

df_video['timestamp_video'] = pd.to_datetime(df_video['timestamp_video'])
df_train_data['timestamp'] = pd.to_datetime(df_train_data['timestamp'])

df_video_merged = pd.merge_asof(
    df_video.sort_values('timestamp_video'),
    df_train_data.sort_values('timestamp'),
    left_on='timestamp_video',
    right_on='timestamp',
    direction='nearest',
    tolerance=pd.Timedelta('10s')
)

df_video_merged = df_video_merged.dropna(subset=['file'])

cols = [c for c in df_video_merged.columns if c not in ["video", "timestamp_video"]] + ["video", "timestamp_video"]
df_video_merged = df_video_merged[cols]

df_video_merged.head(5)

,file,timestamp_raw,timestamp,timestamp_serie,delta_t,velocity,length,total_length,first_peak_raw,video,timestamp_video
190,ADTC_8_adz_raw_20260304_1430,2026-03-04 14:23:16.387,2026-03-04 14:23:26.094,"[Timestamp('2026-03-04 14:23:26.094000'), Time...","[0.376, 0.137, 0.399, 0.139, 0.251, 0.131, 0.9...","[19.61236039465117, 19.752835790595576, 19.422...","[7.37424750838884, 2.706138503311594, 7.749599...",101.366520,9.929712,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 14:23:27
194,ADTC_8_adz_raw_20260304_1440,2026-03-04 14:37:24.352,2026-03-04 14:37:34.398,"[Timestamp('2026-03-04 14:37:34.398000'), Time...","[0.105, 0.702, 0.105, 0.199, 0.11, 0.733, 0.11...","[22.776089159067883, 22.74894290864372, 22.018...","[2.391489361702128, 15.96975792186789, 2.31197...",96.864900,9.929712,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 14:37:35
196,ADTC_8_adz_raw_20260304_1500,2026-03-04 14:54:10.352,2026-03-04 14:54:20.401,"[Timestamp('2026-03-04 14:54:20.401000'), Time...","[0.117, 0.78, 0.121, 0.227, 0.125, 0.813, 0.12...","[20.645256770756255, 20.494873547505122, 19.83...","[2.415495042178482, 15.986001367053996, 2.4002...",96.151421,9.929712,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 14:54:21
200,ADTC_8_adz_raw_20260304_1510,2026-03-04 15:07:10.432,2026-03-04 15:07:20.484,"[Timestamp('2026-03-04 15:07:20.484000'), Time...","[0.106, 0.75, 0.112, 0.215, 0.117, 0.775, 0.11...","[21.793416572077184, 21.50055991041433, 21.029...","[2.3101021566401814, 16.12541993281075, 2.3553...",96.783406,9.929712,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 15:07:21
201,ADTC_8_adz_raw_20260304_1530,2026-03-04 15:23:08.365,2026-03-04 15:23:18.458,"[Timestamp('2026-03-04 15:23:18.458000'), Time...","[0.118, 0.352, 0.116, 0.228, 0.108, 0.825, 0.1...","[22.38200048911714, 22.253021756647865, 22.228...","[2.6410760577158223, 7.833063658340048, 2.5784...",95.631614,9.929712,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 15:23:14


In [34]:
import csv
import os

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

src = root / f'VIDEO/{datum}/'

rest = []

for f in os.listdir(src):
    if f == 'reference_video.csv':
        continue
    file = os.path.join(src, f)

    if file not in df_video_merged['video'].values:
        os.remove(file)
    else:
        rest.append(file)
    
output_csv = os.path.join(src, 'reference_video.csv')

if os.path.exists(output_csv):
    print('reference_video.csv already exists')
    
else:
    with open(output_csv, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile, delimiter=';')  # use ; as separator
        writer.writerow(['video', 'train_type'])
        for vid in rest:
            writer.writerow([vid, ''])

# Manually assign the train type in the CSV file

In [35]:
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

csv_train = root / f'VIDEO/{datum}/reference_video.csv'
df_ref = pd.read_csv(csv_train, sep=';')
df_video_merged = df_video_merged.merge(df_ref, on='video', how='left')

df_video_merged.head(5)

,file,timestamp_raw,timestamp,timestamp_serie,delta_t,velocity,length,total_length,first_peak_raw,video,timestamp_video,train_type
0,ADTC_8_adz_raw_20260304_1430,2026-03-04 14:23:16.387,2026-03-04 14:23:26.094,"[Timestamp('2026-03-04 14:23:26.094000'), Time...","[0.376, 0.137, 0.399, 0.139, 0.251, 0.131, 0.9...","[19.61236039465117, 19.752835790595576, 19.422...","[7.37424750838884, 2.706138503311594, 7.749599...",101.366520,9.929712,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 14:23:27,DPZ
1,ADTC_8_adz_raw_20260304_1440,2026-03-04 14:37:24.352,2026-03-04 14:37:34.398,"[Timestamp('2026-03-04 14:37:34.398000'), Time...","[0.105, 0.702, 0.105, 0.199, 0.11, 0.733, 0.11...","[22.776089159067883, 22.74894290864372, 22.018...","[2.391489361702128, 15.96975792186789, 2.31197...",96.864900,9.929712,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 14:37:35,DTZ
2,ADTC_8_adz_raw_20260304_1500,2026-03-04 14:54:10.352,2026-03-04 14:54:20.401,"[Timestamp('2026-03-04 14:54:20.401000'), Time...","[0.117, 0.78, 0.121, 0.227, 0.125, 0.813, 0.12...","[20.645256770756255, 20.494873547505122, 19.83...","[2.415495042178482, 15.986001367053996, 2.4002...",96.151421,9.929712,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 14:54:21,DTZ
3,ADTC_8_adz_raw_20260304_1510,2026-03-04 15:07:10.432,2026-03-04 15:07:20.484,"[Timestamp('2026-03-04 15:07:20.484000'), Time...","[0.106, 0.75, 0.112, 0.215, 0.117, 0.775, 0.11...","[21.793416572077184, 21.50055991041433, 21.029...","[2.3101021566401814, 16.12541993281075, 2.3553...",96.783406,9.929712,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 15:07:21,DTZ
4,ADTC_8_adz_raw_20260304_1530,2026-03-04 15:23:08.365,2026-03-04 15:23:18.458,"[Timestamp('2026-03-04 15:23:18.458000'), Time...","[0.118, 0.352, 0.116, 0.228, 0.108, 0.825, 0.1...","[22.38200048911714, 22.253021756647865, 22.228...","[2.6410760577158223, 7.833063658340048, 2.5784...",95.631614,9.929712,c:\git\RailwAI\VIDEO\20260304\ADTC Rueschlikon...,2026-03-04 15:23:14,DPZ


In [36]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pywt
from PIL import Image

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

path = root / f'CSV_DATA/{filename}_ext'

target_length = 400
n_length = 200

for file in os.listdir(path):
    file_path = os.path.join(path, file)
    df = pd.read_csv(file_path)

    file_name = file.split('_ext')[0]
    match = df_video_merged[df_video_merged['file'] == file_name]

    if match.empty:
        print(f'No match for {file_name}')
        continue

    zugtyp = match['train_type'].iloc[0]
    if zugtyp == 'DTZ':
        classes = 'Triebzug'
    elif zugtyp == 'DPZ':
        classes = 'Lokzug'
    else:
        classes = 'Other'

    time = df['Time [s]'].values
    length = df['Length [m]'].values
    signal = df['Distance[mm]'].values

    # # shift to zero / clean the data to be sure
    # time = time - time[0]
    # length = length - length[0]
    # mask = np.isfinite(time) & np.isfinite(signal) & np.isfinite(length)

    # time = time[mask]
    # length = length[mask]
    # signal = signal[mask]

    # Downsampling - to speed up CWT
    decim = 5
    time_ds = time[::decim]
    length_ds = length[::decim]
    signal_ds = signal[::decim]

    # -------------------------------------------
    #    #######        ##      ##      ########
    #   ##              ##      ##         ##  
    #   ##              ##      ##         ##  
    #   ##              ##  ##  ##         ##
    #   ##              ##  ##  ##         ##
    #    #######         ###  ###          ## 
    # ------------------------------------------

    # Continuous Wavelet Transform
    dt = np.mean(np.diff(time_ds))                  # time step
    if not np.isfinite(dt) or dt <= 0:
        continue
    fs = 1 / dt                                     # frequency
    freqs = np.linspace(0.2, 5, 100)                # Define frequency range
    cf = pywt.central_frequency('cmor1.5-1.0')      # CWT using Morlet wavelet
    scales = cf * fs / freqs                        # convert frequency to a scale
    scales = np.clip(scales, 1, 1000)
    coeffs, _ = pywt.cwt(                           # Compute CWT
        signal_ds,
        scales,
        'cmor1.5-1.0',
        sampling_period=dt
    )
    power = np.abs(coeffs) ** 2                     # power spectrum

    # Normalize length
    length_ds = length_ds - length_ds.min()
    if length_ds.max() <= 0:
        continue
    length_ds = length_ds / length_ds.max() * target_length

    # fixed grid
    x_out = np.linspace(0, target_length, n_length) # images with 200 pixels width
    power_resampled = np.zeros((power.shape[0], n_length))
    max_len = length_ds[-1]
    max_idx = np.searchsorted(x_out, max_len)
    if max_idx < 2:
        continue

    # interpolation
    for i in range(power.shape[0]):
        power_resampled[i, :max_idx] = np.interp( # resample wavelet result
            x_out[:max_idx],
            length_ds,
            power[i, :]
        )

    # generate plots (images)
    fig, ax1 = plt.subplots(figsize=(5.12, 5.12), dpi=100)
    fig.patch.set_facecolor('white')
    ax1.set_facecolor('white')
    ax1.imshow(
        power_resampled,
        extent=[0, target_length, freqs[0], freqs[-1]],
        aspect='auto',
        cmap='Greys_r',
        origin='lower'
    )
    ax1.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    # save
    out_path = root / f'images/AI-training/{classes}'
    os.makedirs(out_path, exist_ok=True)
    out_filename = os.path.splitext(os.path.basename(file_path))[0] + ".png"
    img_path = os.path.join(out_path, out_filename)
    fig.savefig(
        img_path,
        dpi=100,
        bbox_inches='tight',
        pad_inches=0,
        facecolor='white'
    )

    # remove alpha channel (RGBA -> RGB)
    Image.open(img_path).convert('RGB').save(img_path)
    plt.close(fig)